# Notebook 03: a more robust CNN, still trained from scratch

**UECS2523 Applied Machine Learning, group assignment**

Notebook 01 showed the problem. Many studio-style images in the dataset sit on a white background, so the models partly learned "white background → battery / trash" instead of the object itself. They also only ever saw large, centred objects, so they fail on a real phone photo such as slippers on a wire rack.

This notebook keeps every rule from notebook 01 (**no pretrained weights, same garbage-12 test split, same training recipe**) and changes the training data and augmentation:

| Upgrade | Aim |
|---|---|
| 1. **Background swap**: cut objects out of white-background photos and paste them onto textures, real room photos and concrete floors | weaken the "white background = class" shortcut |
| 2. **Zoom-out and move**, for every class: masked objects are shrunk and moved; images without a usable mask are shrunk as a whole picture | real photos show small, off-centre objects |
| 3. **Extra datasets** with real photos (copies of our own images removed first) | more variety, real waste |
| 4. **Our own phone photos** in training (other items, other rooms than the real-life test) | the conditions we care about |

**How the effect is measured, honestly:**
* The garbage-12 test split is unchanged, so "before" (notebook 01's CNN-B and FCNN-C, rebuilt exactly) and "after" (CNN-R, FCNN-R) are like-for-like there.
* **Ablations** separate the upgrades: CNN-R-noswap (no background swap) and CNN-R-noown (no own photos).
* A **matched-geometry stress test** pastes the same test objects, with the same size and position, once on their own background colour and once on held-out backgrounds from categories never used in training. The difference (the *background gap*, after Xiao et al., 2021) measures background dependence.
* The extra datasets' test splits are **in-domain for the new models** (they trained on the same sources), so they are reported but not claimed as robustness.
* The real test is a fresh set of phone photos in notebook 02, taken in places our training photos never show.

**Run notebook 01 first** (same `QUICK_RUN` setting). This notebook reads its `split.csv`, saved weights and exported models from Google Drive.

## 1. Setup

In [ ]:
# ---- Settings you may want to change --------------------------------------
QUICK_RUN = False           # True = tiny subset + 2 epochs, to check the pipeline (needs a QUICK_RUN of notebook 01)
USE_DRIVE = True
SKIP_TRAINED = True
PROJECT_NAME = "AML_Waste_Project"
KAGGLE_DATASET = "mostafaabla/garbage-classification"
DATASET_DIR_OVERRIDE = None

IMG_SIZE = 128
BATCH_SIZE = 64
SEED = 42
LABEL_SMOOTHING = 0.1
MAX_EPOCHS = 80              # same recipe as notebook 01 (cap 80, patience 12, LR patience 4)
EARLY_STOP_PATIENCE = 12

USE_EXTRA_DATASETS = True    # Part 3
USE_OWN_TRAIN_PHOTOS = True  # Part 4: MyDrive/AML_Waste_Project/train_photos/<class>/
OWN_PHOTO_REPEAT = 3         # each own photo appears this many times per epoch
USE_BACKGROUND_SWAP = True   # Part 6
COMPOSITE_SHARE = 0.35       # in every class, about this share of training images is pasted onto a new background per epoch
SCALE_RANGE = (0.4, 1.0)     # pasted object size relative to the frame
USE_DTD = True               # texture photos (Describable Textures Dataset, about 600 MB download)
USE_INDOOR_SCENES = True     # real room photos (MIT Indoor-67 mirror, about 140 MB download)
RUN_ABLATIONS = True         # CNN-R-noswap and CNN-R-noown (about 1-1.5 extra T4 hours; needed for the report's claims)
TRAIN_RESCNN = False         # optional larger residual CNN (about 1 more hour)
# ---------------------------------------------------------------------------

In [ ]:
import os, sys, io, json, time, hashlib, shutil, warnings, zipfile, tarfile, urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from scipy import ndimage
from tqdm.auto import tqdm
from IPython.display import display

import tensorflow as tf
import keras
from keras import layers

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score
from scipy.stats import binomtest

warnings.filterwarnings("ignore", category=UserWarning)
tf.get_logger().setLevel("ERROR")
Image.MAX_IMAGE_PIXELS = 200_000_000
RESAMPLE = getattr(Image, "Resampling", Image).LANCZOS
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print(f"TensorFlow {tf.__version__} | Keras {keras.__version__} | Colab: {IN_COLAB}")

GPUS = tf.config.list_physical_devices("GPU")
for g in GPUS:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except RuntimeError:
        pass
if GPUS:
    GPU_NAME = tf.config.experimental.get_device_details(GPUS[0]).get("device_name", "GPU")
    keras.mixed_precision.set_global_policy("mixed_float16")
    print(f"GPU found: {GPU_NAME}. Mixed precision ON.")
else:
    GPU_NAME = "CPU only"
    print("No GPU found. In Colab: Runtime > Change runtime type > T4 GPU, then run all cells again.")
keras.utils.set_random_seed(SEED)

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME
else:
    OUT_DIR = Path.cwd() / PROJECT_NAME
PROJECT_DIR = OUT_DIR                      # real_photos/ and train_photos/ always live here
if QUICK_RUN:
    OUT_DIR = OUT_DIR / "quick_run"
LOGS_01, EXPORT_DIR, MODELS_01 = OUT_DIR / "logs", OUT_DIR / "export", OUT_DIR / "models"
ROBUST_DIR = OUT_DIR / "robust"
MODELS_DIR, LOGS_DIR, FIG_DIR = (ROBUST_DIR / d for d in ("models", "logs", "figures"))
for d in (MODELS_DIR, LOGS_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
WORK = Path("/content/extra_data") if IN_COLAB else Path.cwd() / "extra_data"   # big downloads stay off Drive
WORK.mkdir(parents=True, exist_ok=True)

needed = [LOGS_01 / "split.csv", EXPORT_DIR / "meta.json", MODELS_01 / "CNN-B.weights.h5", MODELS_01 / "FCNN-C.weights.h5"]
missing = [str(p) for p in needed if not p.exists()]
if missing:
    raise FileNotFoundError(f"Run notebook 01 first (QUICK_RUN = {QUICK_RUN}). Missing: {missing}")
META = json.loads((EXPORT_DIR / "meta.json").read_text())
CLASS_NAMES = META["class_names"]
NUM_CLASSES = len(CLASS_NAMES)
class_to_id = {c: i for i, c in enumerate(CLASS_NAMES)}
print("Outputs go to:", ROBUST_DIR)


def save_fig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight", dpi=150)

### Image reading (identical to notebook 01, plus one extra fingerprint)
EXIF rotation fix, transparent areas → white, centre square crop, resize to 128 × 128. Besides the MD5 hash, difference hash and white-border share from notebook 01, each image also gets an *aspect-free* 32 × 32 grey thumbnail of the whole picture. It catches copies that were squashed, padded or re-coloured, which the centre-crop check misses.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp", ".tif", ".tiff"}


def to_rgb(im):
    if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
        im = im.convert("RGBA")
        bg = Image.new("RGB", im.size, (255, 255, 255))
        bg.paste(im, mask=im.getchannel("A"))
        return bg
    return im.convert("RGB")


def center_crop_resize(im, size):
    w, h = im.size
    s = min(w, h)
    left, top = (w - s) // 2, (h - s) // 2
    return im.crop((left, top, left + s, top + s)).resize((size, size), RESAMPLE)


def dhash(im, hash_size=8):
    g = np.asarray(im.convert("L").resize((hash_size + 1, hash_size), RESAMPLE), dtype=np.int16)
    return np.packbits((g[:, 1:] > g[:, :-1]).flatten()).tobytes().hex()


def border_white_fraction(arr, frac=0.1):
    s = arr.shape[0]
    b = max(1, int(s * frac))
    mask = np.ones((s, s), bool)
    mask[b:-b, b:-b] = False
    return float((arr[mask].min(axis=1) > 230).mean())


def read_image(path):
    raw = Path(path).read_bytes()
    info = {"md5": hashlib.md5(raw).hexdigest()}
    with Image.open(io.BytesIO(raw)) as im:
        im.load()
        im = to_rgb(ImageOps.exif_transpose(im))
    info["dhash"] = dhash(im)
    g = np.asarray(im.convert("L").resize((32, 32), RESAMPLE), np.float32).ravel()
    info["ft"] = ((g - g.mean()) / (g.std() + 1e-6)).astype(np.float16).tobytes()     # aspect-free thumbnail
    arr = np.asarray(center_crop_resize(im, IMG_SIZE), dtype=np.uint8)
    info["white_border"] = border_white_fraction(arr)
    return info, arr


def read_many(paths, desc):
    X = np.zeros((len(paths), IMG_SIZE, IMG_SIZE, 3), np.uint8)
    recs = [None] * len(paths)

    def work(i):
        try:
            info, arr = read_image(paths[i])
            X[i] = arr
            info["ok"] = True
        except Exception as e:
            info = {"ok": False, "error": f"{type(e).__name__}: {e}"}
        recs[i] = info

    with ThreadPoolExecutor(max_workers=8) as ex:
        list(tqdm(ex.map(work, range(len(paths))), total=len(paths), desc=desc))
    return X, pd.DataFrame(recs)


def collect_files(base, mapping, include=None):
    """Every image in a folder whose name (case-insensitive) is a key of `mapping`; value = our class or None (skip).
    include: only keep files that have this folder name somewhere in their path (e.g. 'original'). Sorted, so the
    result does not depend on the order the file system returns folders in."""
    keys = {k.lower(): v for k, v in mapping.items()}
    files, labels = [], []
    for dirpath, dirnames, filenames in os.walk(base):
        dirnames.sort()
        cls = keys.get(Path(dirpath).name.lower(), None)
        if cls is None:
            continue
        if include and include.lower() not in [part.lower() for part in Path(dirpath).parts]:
            continue
        for f in sorted(filenames):
            if Path(f).suffix.lower() in IMG_EXTS:
                files.append(os.path.join(dirpath, f)); labels.append(cls)
    return files, labels


def channel_stats(x, chunk=2048):
    s = np.zeros(3); ss = np.zeros(3); n = 0
    for i in range(0, len(x), chunk):
        b = x[i:i + chunk].astype(np.float64).reshape(-1, 3)
        s += b.sum(0); ss += (b ** 2).sum(0); n += len(b)
    m = s / n
    return m, ss / n - m ** 2

### Duplicate detection
Two rules, both from notebook 01 or verified on this dataset:
* **near_pairs**: difference hash within 4 bits *and* matching centre-crop thumbnails (small mean difference, correlation above 0.95). Precise, catches re-encoded and resized copies.
* **copy_pairs**: difference hash within 8 bits *and* aspect-free whole-image thumbnails correlated above 0.90. Also catches squashed, padded and re-coloured copies.

Against garbage-12 we remove an extra image if *either* rule fires, because a missed copy of a test image would leak the test set, while a false alarm only costs one extra image. Inside the extras, near-duplicate *groups* (kept in one split) use only the strict rule: on garbage-12 the loose rule chains look-alike images into groups of 100+, which would push whole chunks of a dataset into one split. Cropped or mirrored copies are still not guaranteed to be found.

In [ ]:
def dhash_bits(hexes):
    return np.unpackbits(np.frombuffer(bytes.fromhex("".join(hexes)), np.uint8).reshape(-1, 8), axis=1).astype(np.float32)


def thumbs(X):
    k = IMG_SIZE // 32
    t = np.concatenate([X[s:s + 2000].reshape(-1, 32, k, 32, k, 3).mean(axis=(2, 4, 5)) for s in range(0, len(X), 2000)]) \
        if len(X) else np.zeros((0, 32, 32), np.float32)
    t = t.reshape(len(X), -1).astype(np.float32)
    z = (t - t.mean(1, keepdims=True)) / (t.std(1, keepdims=True) + 1e-6)
    return t, z


def full_thumbs(df):
    return np.stack([np.frombuffer(b, np.float16) for b in df["ft"]]).astype(np.float32) if len(df) else np.zeros((0, 1024), np.float32)


def near_pairs(bits_a, thumb_a, z_a, bits_b, thumb_b, z_b, same=False):
    ones_a, ones_b = bits_a.sum(1), bits_b.sum(1)
    for s in range(0, len(bits_a), 2000):
        ham = ones_a[s:s + 2000, None] + ones_b[None, :] - 2 * bits_a[s:s + 2000] @ bits_b.T
        for i, j in zip(*np.nonzero(ham <= 4)):
            i += s
            if same and i >= j:
                continue
            if np.abs(thumb_a[i] - thumb_b[j]).mean() < 8 and float(z_a[i] @ z_b[j]) / z_a.shape[1] > 0.95:
                yield i, j


def copy_pairs(bits_a, ft_a, bits_b, ft_b, same=False, max_ham=8, min_r=0.90, block=1000):
    oa, ob = bits_a.sum(1), bits_b.sum(1)
    for s in range(0, len(bits_a), block):
        ham = oa[s:s + block, None] + ob[None, :] - 2 * bits_a[s:s + block] @ bits_b.T
        r = ft_a[s:s + block] @ ft_b.T / ft_a.shape[1]
        for i, j in zip(*np.nonzero((ham <= max_ham) & (r > min_r))):
            if not same or i + s < j:
                yield i + s, j

## 2. The original dataset, with notebook 01's exact split

In [ ]:
def kaggle_download(slug):
    try:
        import kagglehub
    except ImportError:
        os.system(f"{sys.executable} -m pip install -q kagglehub")
        import kagglehub
    try:
        return Path(kagglehub.dataset_download(slug))
    except Exception as first_error:
        print("Anonymous download failed:", first_error)
        if IN_COLAB:
            from google.colab import userdata
            try:
                os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
                os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
            except Exception as secret_error:
                raise RuntimeError("Add KAGGLE_USERNAME and KAGGLE_KEY as Colab secrets "
                                   "(key icon on the left) and run this cell again.") from secret_error
        return Path(kagglehub.dataset_download(slug))


DATA_BASE = Path(DATASET_DIR_OVERRIDE) if DATASET_DIR_OVERRIDE else kaggle_download(KAGGLE_DATASET)
base = pd.read_csv(LOGS_01 / "split.csv")
base["source"] = "garbage-12"
X_base, rec = read_many([str(DATA_BASE / p) for p in base["path"]], "garbage-12 images")
assert rec["ok"].all(), "Some garbage-12 files could not be read; was the dataset re-downloaded correctly?"
base["ft"] = rec["ft"].values
B_MEAN, B_VAR = channel_stats(X_base[(base["split"] == "train").values])     # notebook 01's normalisation
display(pd.crosstab(base["label"], base["split"])[["train", "val", "test"]])

## 3. Extra datasets with real-world photos

Chosen after checking every candidate for overlap with our data, licence and image style:
* **RealWaste** (Single et al., 2023): 4,752 real photos of dirty, crushed landfill waste taken at a recycling facility, none of which appear in our dataset. Licence CC BY 4.0 (UCI) / CC BY-NC-SA 4.0 (readme): fine for a student project. Mapping rule: textile waste counts as clothes (textile recycling) and garden waste counts as biological (compost). Its single "glass" class cannot be split by colour and is skipped. All its photos are top-down on grey concrete, so concrete floor patches are also used as backgrounds for every class (Part 6).
* **Garbage Classification v2** (sumn2u, MIT licence): about 70% of its images are byte-identical copies of our dataset and every image is stored three times, so only its `original/` folder is used and the de-duplication below removes the copies. About 2,900 new images remain (its "glass" class is skipped).

Rejected: TrashNet copies (already inside our dataset), TrashBox (no licence, 2.3 GB, noisy web images), and several Kaggle sets with conflicting licences or many duplicates.

A source can be `"kaggle"` (a Kaggle slug), `"url"` (a zip/tar link) or `"local"` (a folder you uploaded yourself). Set `USE_EXTRA_DATASETS = False` in Part 1 to skip them all.

**The split of the extras is saved** (by MD5) on the first run and reused afterwards, so a model trained in one Colab session is never tested on its own training images in the next.

In [ ]:
EXTRA_SOURCES = [
    {"name": "RealWaste", "kaggle": "joebeachcapital/realwaste",
     "url_fallback": "https://archive.ics.uci.edu/static/public/908/realwaste.zip",
     "map": {"Cardboard": "cardboard", "Food Organics": "biological", "Vegetation": "biological", "Metal": "metal",
             "Paper": "paper", "Plastic": "plastic", "Miscellaneous Trash": "trash", "Textile Trash": "clothes",
             "Glass": None}},
    {"name": "Garbage-v2", "kaggle": "sumn2u/garbage-classification-v2", "include": "original",
     "map": {c: c for c in ("battery", "biological", "cardboard", "clothes", "metal", "paper", "plastic", "shoes", "trash")}
            | {"glass": None}},
]
EXTRA_SOURCES = [s for s in EXTRA_SOURCES if all(v is None or v in class_to_id for v in s["map"].values())]


def url_download(url, name):
    """Download + extract once per VM; a broken or partial download is deleted so the next run retries."""
    target = WORK / name
    marker = target / ".done"
    if marker.exists():
        return target
    target.mkdir(parents=True, exist_ok=True)
    archive = WORK / f"{name}__{Path(url.split('?')[0]).name}"
    if not archive.exists():
        print("Downloading", url)
        tmp = archive.with_name(archive.name + ".part")
        urllib.request.urlretrieve(url, tmp)
        tmp.replace(archive)
    try:
        if zipfile.is_zipfile(archive):
            with zipfile.ZipFile(archive) as z:
                z.extractall(target)
        else:
            with tarfile.open(archive) as t:
                t.extractall(target)
    except Exception:
        archive.unlink(missing_ok=True)
        raise
    marker.touch()
    return target


ext_frames, ext_arrays = [], []
if USE_EXTRA_DATASETS:
    for src in EXTRA_SOURCES:
        root = None
        for kind in ("local", "kaggle", "url", "url_fallback"):
            if kind not in src:
                continue
            try:
                root = (Path(src[kind]) if kind == "local" else kaggle_download(src[kind]) if kind == "kaggle"
                        else url_download(src[kind], src["name"]))
                break
            except Exception as e:
                print(f"{src['name']}: {kind} download failed ({e})")
        if root is None:
            print(f"Skipping {src['name']}")
            continue
        files, labels = collect_files(root, src["map"], src.get("include"))
        if not files:
            print(f"Skipping {src['name']}: no mapped class folders found under {root}")
            continue
        df = pd.DataFrame({"path": files, "label": labels, "source": src["name"],
                           "folder": [Path(f).parent.name for f in files]})
        if QUICK_RUN:
            df = pd.concat([g.sample(min(len(g), 40), random_state=SEED) for _, g in df.groupby("label")]).reset_index(drop=True)
        Xs, recs = read_many(df["path"].tolist(), src["name"])
        df = pd.concat([df, recs], axis=1)
        keep = df["ok"].values
        ext_frames.append(df[keep].reset_index(drop=True)); ext_arrays.append(Xs[keep])
        print(f"{src['name']}: {keep.sum():,} images read ({(~keep).sum()} unreadable)")
        del Xs, recs

In [ ]:
if ext_frames:
    ext = pd.concat(ext_frames, ignore_index=True)
    X_ext = np.concatenate(ext_arrays)
    del ext_frames, ext_arrays
    audit = {"extra images read": len(ext)}

    drop = ext["md5"].isin(set(base["md5"])).values
    audit["exact copies of garbage-12 images (removed)"] = int(drop.sum())
    md5_lab = ext[~drop].groupby("md5")["label"].nunique()
    conflict = ext["md5"].isin(set(md5_lab[md5_lab > 1].index)).values
    dup = ext.duplicated("md5", keep="first").values
    audit["exact duplicates inside the extras (removed)"] = int(((dup | conflict) & ~drop).sum())
    drop |= dup | conflict

    bb, (tb, zb), fb = dhash_bits(base["dhash"]), thumbs(X_base), full_thumbs(base)
    be, (te, ze), fe = dhash_bits(ext["dhash"]), thumbs(X_ext), full_thumbs(ext)
    near = np.zeros(len(ext), bool)
    for i, _ in near_pairs(be, te, ze, bb, tb, zb):
        near[i] = True
    for i, _ in copy_pairs(be, fe, bb, fb):
        near[i] = True
    audit["near-copies of garbage-12 images (removed)"] = int((near & ~drop).sum())
    drop |= near
    del tb, zb, fb

    keep = ~drop
    ext = ext[keep].reset_index(drop=True)
    X_ext, be, te, ze, fe = X_ext[keep], be[keep], te[keep], ze[keep], fe[keep]

    parent = np.arange(len(ext))

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    for i, j in near_pairs(be, te, ze, be, te, ze, same=True):      # strict rule only (see above)
        parent[find(i)] = find(j)
    ext["group"] = [f"x{find(i)}" for i in range(len(ext))]
    audit["largest near-duplicate group inside the extras"] = int(ext["group"].value_counts().max()) if len(ext) else 0
    del be, te, ze, fe

    split_file = ROBUST_DIR / "extra_split_by_md5.csv"
    if split_file.exists():                                  # reuse the split from the first session
        saved = pd.read_csv(split_file).set_index("md5")["split"]
        ext["split"] = ext["md5"].map(saved)
        new = ext["split"].isna().values
        audit["images not in the saved split (dropped)"] = int(new.sum())
        ext, X_ext = ext[~new].reset_index(drop=True), X_ext[~new]
        print(f"Reusing the extras split saved in {split_file.name}")
    else:
        glab = ext.groupby("group").agg(label=("label", lambda s: s.mode().iloc[0]), source=("source", "first"))
        def make_strata(g, min_count=7):
            """source|label, with tiny classes pooled per source; a pool that is still too small (e.g. Garbage-v2
            has only ONE new clothes image) joins that source's biggest class, because a stratified split
            needs at least 2 members per stratum."""
            st = g["source"] + "|" + g["label"]
            small = st.map(st.value_counts()) < min_count
            st[small] = g.loc[small, "source"] + "|rare"
            counts = st.value_counts()
            for lab in counts[counts < 2].index:
                src_ = lab.split("|")[0]
                same_src = counts[[k for k in counts.index if k.startswith(src_ + "|") and k != lab]]
                st[st == lab] = same_src.idxmax() if len(same_src) else counts.idxmax()
            return st

        def safe_split(ids, test_size):
            try:
                return train_test_split(ids, test_size=test_size, stratify=np.asarray(make_strata(glab.loc[ids])),
                                        random_state=SEED)
            except ValueError as e:            # very small data (e.g. QUICK_RUN): fall back to a plain random split
                print("Stratified split not possible, using a random split:", e)
                return train_test_split(ids, test_size=test_size, random_state=SEED)

        g_tv, g_te = safe_split(np.asarray(glab.index, dtype=object), 0.15)
        g_tr, g_va = safe_split(g_tv, 0.15 / 0.85)
        split_of = {**{g: "train" for g in g_tr}, **{g: "val" for g in g_va}, **{g: "test" for g in g_te}}
        ext["split"] = ext["group"].map(split_of)
        ext[["md5", "source", "label", "split"]].to_csv(split_file, index=False)
    ext["y"] = ext["label"].map(class_to_id)
    audit["extra images kept"] = len(ext)
    ext.drop(columns=["path", "ft"]).to_csv(LOGS_DIR / "extra_split.csv", index=False)
    display(pd.DataFrame.from_dict(audit, orient="index", columns=["count"]))
    display(pd.crosstab([ext["source"], ext["label"]], ext["split"]))
else:
    ext = pd.DataFrame(columns=["label", "source", "split", "y", "white_border", "md5", "path", "folder"])
    X_ext = np.zeros((0, IMG_SIZE, IMG_SIZE, 3), np.uint8)
    print("No extra datasets used.")

**Check the mapping by eye.** Eight random images from every mapped RealWaste folder. If a folder holds items that clearly belong to another class (for example a shoe in "Textile Trash"), note it in the report.

In [ ]:
rw = ext[ext["source"] == "RealWaste"] if len(ext) else ext
if len(rw):
    folders = sorted(rw["folder"].unique())
    fig, axes = plt.subplots(len(folders), 8, figsize=(13, 1.8 * len(folders)), squeeze=False)
    for r, fo in enumerate(folders):
        idx = rw.index[rw["folder"] == fo].values
        pick = np.random.default_rng(SEED).choice(idx, min(8, len(idx)), replace=False)
        for c in range(8):
            axes[r, c].axis("off")
            if c < len(pick):
                axes[r, c].imshow(X_ext[pick[c]])
        axes[r, 0].set_title(f"{fo} → {rw.loc[idx[0], 'label']}", fontsize=8, loc="left")
    plt.tight_layout(); save_fig(fig, "realwaste_mapping"); plt.show()

## 4. Our own phone photos for training

Put photos in `MyDrive/AML_Waste_Project/train_photos/<class>/`, one folder per class (created below).

**Keep them separate from the real-life test set (`real_photos/`, notebook 02):** different physical items **and** different rooms, surfaces and days, ideally taken by a different group member. Otherwise the robust model could simply learn our rooms, and its real-photo score would be inflated. Copies and near-copies of test photos are removed automatically, but only you can keep the items and places apart.

20 to 30 photos per class is plenty. Each photo is shown `OWN_PHOTO_REPEAT` times per epoch. The ablation CNN-R-noown is trained without them, so their effect can be measured on its own.

In [ ]:
try:  # iPhone HEIC photos
    from pillow_heif import register_heif_opener
except ImportError:
    os.system("pip install -q pillow-heif")
    try:
        from pillow_heif import register_heif_opener
    except ImportError:
        register_heif_opener = None
if register_heif_opener:
    register_heif_opener()
IMG_EXTS |= {".heic", ".heif"}

OWN_DIR = PROJECT_DIR / "train_photos"
for c in CLASS_NAMES:
    (OWN_DIR / c).mkdir(parents=True, exist_ok=True)
own_files, own_labels = [], []
if USE_OWN_TRAIN_PHOTOS:
    for c in CLASS_NAMES:
        for p in sorted((OWN_DIR / c).glob("*")):
            if p.suffix.lower() in IMG_EXTS:
                own_files.append(str(p)); own_labels.append(c)
own = pd.DataFrame({"label": own_labels, "source": "own photos"})
X_own = np.zeros((0, IMG_SIZE, IMG_SIZE, 3), np.uint8)
if len(own):
    X_own, recs = read_many(own_files, "own photos")
    own = pd.concat([own, recs], axis=1)
    own, X_own = own[own["ok"].values].reset_index(drop=True), X_own[own["ok"].values]
    test_files = sorted(str(p) for p in (PROJECT_DIR / "real_photos").rglob("*") if p.suffix.lower() in IMG_EXTS)
    clash = np.zeros(len(own), bool)
    if test_files:
        X_tp, tp = read_many(test_files, "real-life test photos (for the copy check)")
        tp = tp[tp["ok"]].reset_index(drop=True)
        clash |= own["md5"].isin(set(tp["md5"])).values
        for i, _ in copy_pairs(dhash_bits(own["dhash"]), full_thumbs(own), dhash_bits(tp["dhash"]), full_thumbs(tp)):
            clash[i] = True
        del X_tp
    if clash.any():
        print(f"Removed {clash.sum()} own training photos that are copies or near-copies of real-life TEST photos.")
    own, X_own = own[~clash].reset_index(drop=True), X_own[~clash]
    own["y"] = own["label"].map(class_to_id)
    own["split"] = "train"
    display(own["label"].value_counts().rename("own training photos").to_frame().T)
else:
    print(f"No own training photos found in {OWN_DIR} (optional).")

## 5. Training, validation and test sets

* **Training**: garbage-12 train + extra train + own photos (repeated).
* **Validation** (early stopping, learning-rate schedule): garbage-12 val + extra val, so training has to work on real photos too. Garbage-12-only validation scores are logged as well.
* **Tests**, opened once at the end: the unchanged garbage-12 test split, each extra source's test split (in-domain for the new models) and the stress tests (Part 7).

In [ ]:
def take(df, X, split):
    m = (df["split"] == split).values if len(df) else np.zeros(0, bool)
    return df[m].reset_index(drop=True), X[m]


frames, arrays = {}, {}
for split in ("train", "val", "test"):
    b_df, b_X = take(base, X_base, split)
    e_df, e_X = take(ext, X_ext, split)
    f_, a_ = [b_df, e_df], [b_X, e_X]
    if split == "train" and len(own):
        f_ += [own] * OWN_PHOTO_REPEAT
        a_ += [X_own] * OWN_PHOTO_REPEAT
    frames[split], arrays[split] = pd.concat(f_, ignore_index=True), np.concatenate(a_)
    del b_X, e_X

train_df, X_train = frames["train"], arrays["train"]
val_df, X_val = frames["val"], arrays["val"]
Y_train, Y_val = train_df["y"].values.astype("int32"), val_df["y"].values.astype("int32")
test_sets = {"garbage-12 test": take(base, X_base, "test")}
for s in (ext["source"].unique() if len(ext) else []):
    sub = (ext["source"] == s).values
    test_sets[f"{s} test (in-domain for 'after')"] = take(ext[sub].reset_index(drop=True), X_ext[sub], "test")
for k in [k for k, v in test_sets.items() if len(v[0]) == 0]:
    print(f"Skipping '{k}': no images left after de-duplication")
    test_sets.pop(k)

# RealWaste empty-floor patches (from TRAIN images only) are used as extra backgrounds in Part 6
rw_train_paths = ext.loc[(ext["source"] == "RealWaste") & (ext["split"] == "train"), "path"].tolist() if len(ext) else []
del X_base, X_ext, X_own, frames, arrays          # free memory: the parts above hold what is needed

norm_file = ROBUST_DIR / "normalisation.json"
fingerprint = hashlib.sha1("".join(sorted(train_df["md5"].astype(str))).encode()).hexdigest()
if norm_file.exists():
    saved = json.loads(norm_file.read_text())
    MEAN, VAR = np.array(saved["mean"]), np.array(saved["var"])
    if saved["fingerprint"] != fingerprint:
        print("WARNING: the training set differs from the one the saved robust models were trained on "
              "(for example new own photos). They will be retrained.")
        SKIP_TRAINED = False
        MEAN, VAR = channel_stats(X_train)
        norm_file.write_text(json.dumps({"mean": MEAN.tolist(), "var": VAR.tolist(), "fingerprint": fingerprint}))
else:
    MEAN, VAR = channel_stats(X_train)
    norm_file.write_text(json.dumps({"mean": MEAN.tolist(), "var": VAR.tolist(), "fingerprint": fingerprint}))

summary = pd.DataFrame({"images": {"train": len(train_df), "val": len(val_df),
                                   **{k: len(v[0]) for k, v in test_sets.items()}}})
display(summary.T)
print(f"Training array: {X_train.nbytes / 2**30:.2f} GB")
display(pd.crosstab(train_df["label"], train_df["source"]))

## 6. Background swap and zoom-out augmentation

**Object masks.** For a training image with a mostly white border, near-white pixels connected to the image edge are background; the rest is the object. Masks that look wrong (object too small or too large, or broken into many pieces, as with newspapers or see-through glass) are not used.

**Every class gets the same treatment.** Masks work mostly for battery, shoes and glass, which would make "pasted onto a texture" a new clue for those classes. So the probability of compositing is set per class to give each class about `COMPOSITE_SHARE` composited images per epoch: objects with a usable mask are cut out; all other images are shrunk and pasted *as a whole picture*. Zoom-out therefore reaches clothes and paper too.

**Backgrounds** (shared by every class, so the background says nothing about the label, as in the "mixed-random" setup of Xiao et al., 2021): texture photos from the Describable Textures Dataset (Cimpoi et al., 2014), crops of real room photos (MIT Indoor-67; scenes full of our own classes such as shoe shops are skipped), empty concrete-floor patches from RealWaste training images (so batteries, shoes and glass also appear on concrete), and procedurally drawn textures. **Whole texture and room categories are held out** for the stress test, so its backgrounds are of kinds never seen in training.

**Blending** is randomised between a soft and a hard edge, so the paste seam itself is not a clue (Dwibedi et al., 2017). Following Patrizi et al. (2021), original images stay in the mix; the remaining white backgrounds come mainly from images whose mask failed.

In [ ]:
def object_alpha(img, white_min=215, sat_max=40, sigma=1.0):
    """Returns (alpha uint8 0-255, usable?) for a white-background image."""
    a = img.astype(np.int16)
    near_white = (a.min(-1) >= white_min) & ((a.max(-1) - a.min(-1)) <= sat_max)
    lab, _ = ndimage.label(near_white)
    border = np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]]))
    obj = ~np.isin(lab, border[border > 0])
    obj = ndimage.binary_fill_holes(ndimage.binary_opening(obj, iterations=1))
    frac = obj.mean()
    lab2, n2 = ndimage.label(obj)
    sizes = np.sort(np.bincount(lab2.ravel())[1:])[::-1] if n2 else np.array([0])
    top2 = sizes[:2].sum() / max(obj.sum(), 1)
    usable = bool(0.08 <= frac <= 0.92 and top2 >= 0.9)     # one or two solid objects
    edge_safe = ndimage.binary_erosion(obj, iterations=1)    # drop the 1-px white fringe so no halo is pasted
    alpha = ndimage.gaussian_filter(edge_safe.astype(np.float32), sigma) * 255
    return alpha.astype(np.uint8), usable


eligible = np.where(train_df["white_border"].fillna(0).values > 0.6)[0]
alpha_row = np.full(len(train_df), -1, np.int32)
ALPHA = np.zeros((0, IMG_SIZE, IMG_SIZE), np.uint8)
if USE_BACKGROUND_SWAP and len(eligible):
    results = [None] * len(eligible)

    def _mask(k):
        results[k] = object_alpha(X_train[eligible[k]])

    with ThreadPoolExecutor(max_workers=8) as ex:
        list(tqdm(ex.map(_mask, range(len(eligible))), total=len(eligible), desc="Object masks"))
    good = [k for k in range(len(eligible)) if results[k][1]]
    ALPHA = np.stack([results[k][0] for k in good]) if good else ALPHA
    alpha_row[eligible[good]] = np.arange(len(good))
    print(f"White-border training images: {len(eligible):,}; usable masks: {len(good):,} "
          f"({len(good) / max(len(eligible), 1):.0%})")
    del results

# per-class compositing probabilities: masked rows use p_mask, the rest p_rect, so every class gets ~COMPOSITE_SHARE
has_mask = alpha_row >= 0
usable_share = pd.Series(has_mask).groupby(Y_train).mean().reindex(range(NUM_CLASSES), fill_value=0).values
p_mask = np.minimum(1.0, COMPOSITE_SHARE / np.maximum(usable_share, 1e-6))
p_rect = np.clip((COMPOSITE_SHARE - usable_share * p_mask) / np.maximum(1 - usable_share, 1e-6), 0, 1)
P_ROW = np.where(has_mask, p_mask[Y_train], p_rect[Y_train]).astype(np.float32)
mix = pd.DataFrame({"usable masks": usable_share, "cut-out per epoch": usable_share * p_mask,
                    "whole-picture paste per epoch": (1 - usable_share) * p_rect,
                    "still white per epoch": pd.Series((1 - P_ROW) * (train_df["white_border"].fillna(0).values > 0.6))
                                             .groupby(Y_train).mean().reindex(range(NUM_CLASSES), fill_value=0).values},
                   index=CLASS_NAMES)
display(mix.T.style.format("{:.0%}"))

In [ ]:
def procedural_textures(n, size=IMG_SIZE, seed=0):
    r = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:size, 0:size].astype(np.float32) / size
    out = np.zeros((n, size, size, 3), np.uint8)
    for i in range(n):
        c1, c2 = r.uniform(0, 255, 3), r.uniform(0, 255, 3)
        kind = r.integers(0, 6)
        if kind == 0:     # gradient
            ang = r.uniform(0, np.pi)
            t = np.cos(ang) * xx + np.sin(ang) * yy
            t = (t - t.min()) / (t.max() - t.min() + 1e-6)
        elif kind == 1:   # wood-like grain
            t = 0.5 + 0.5 * np.sin(r.uniform(8, 40) * (yy + 0.05 * np.sin(r.uniform(3, 9) * xx)) * np.pi)
        elif kind == 2:   # tiles / wire mesh
            k, w = r.integers(4, 24), r.uniform(0.04, 0.25)
            t = (((xx * k) % 1 < w) | ((yy * k) % 1 < w)).astype(np.float32)
        elif kind == 3:   # checker
            k = r.integers(2, 10)
            t = (((xx * k).astype(int) + (yy * k).astype(int)) % 2).astype(np.float32)
        elif kind == 4:   # blotches
            low = r.uniform(0, 1, (int(r.integers(3, 9)),) * 2)
            t = np.asarray(Image.fromarray((low * 255).astype(np.uint8)).resize((size, size), Image.BICUBIC), np.float32) / 255
        else:             # plain colour with grain (concrete-like when grey)
            t = np.zeros((size, size), np.float32)
        img = c1 * (1 - t[..., None]) + c2 * t[..., None] + r.normal(0, r.uniform(2, 18), (size, size, 3))
        out[i] = np.clip(img, 0, 255)
    return out


DTD_URL = "https://thor.robots.ox.ac.uk/dtd/dtd-r1.0.1.tar.gz"     # (Cimpoi et al., 2014), research use
INDOOR_URLS = ["https://huggingface.co/datasets/keremberke/indoor-scene-classification/resolve/main/data/valid.zip",
               "https://huggingface.co/datasets/keremberke/indoor-scene-classification/resolve/main/data/test.zip"]
INDOOR_SKIP = {"shoeshop", "clothingstore", "closet", "laundromat", "winecellar", "bookstore", "grocerystore",
               "bakery", "deli", "fastfood_restaurant", "toystore", "florist", "greenhouse", "jewelleryshop", "videostore",
               "library", "bar", "pantry", "kitchen", "restaurant_kitchen", "warehouse"}


def random_crops(paths, size=IMG_SIZE, seed=0, max_n=3000):
    """One random square crop per photo, at a random zoom, resized to size x size."""
    r = np.random.default_rng(seed)
    paths = [paths[i] for i in r.permutation(len(paths))[:max_n]]
    out = np.zeros((len(paths), size, size, 3), np.uint8)
    ok = np.zeros(len(paths), bool)
    seeds = r.integers(0, 2**31, len(paths))

    def work(i):
        try:
            rr = np.random.default_rng(seeds[i])
            im = to_rgb(Image.open(paths[i]))
            w, h = im.size
            side = int(min(w, h) * rr.uniform(0.35, 1.0))
            x, y = int(rr.integers(0, w - side + 1)), int(rr.integers(0, h - side + 1))
            out[i] = np.asarray(im.crop((x, y, x + side, y + side)).resize((size, size), RESAMPLE))
            ok[i] = True
        except Exception:
            pass

    with ThreadPoolExecutor(max_workers=8) as ex:
        list(ex.map(work, range(len(paths))))
    return out[ok]


def split_by_category(files, test_share=0.2, seed=SEED):
    """Hold out whole categories (parent folder) for the stress test."""
    cats = sorted({Path(f).parent.name for f in files})
    test_cats = set(np.random.default_rng(seed).permutation(cats)[:max(1, int(len(cats) * test_share))])
    return [f for f in files if Path(f).parent.name not in test_cats], [f for f in files if Path(f).parent.name in test_cats]


def corner_patches(paths, size=IMG_SIZE, frac=0.25):
    """Empty corners of RealWaste photos (concrete floor), so every class can appear on concrete."""
    out = []
    for p in paths:
        try:
            im = to_rgb(ImageOps.exif_transpose(Image.open(p)))
            w, h = im.size
            s = int(min(w, h) * frac)
            out += [np.asarray(im.crop((x, y, x + s, y + s)).resize((size, size), RESAMPLE))
                    for x, y in ((0, 0), (w - s, 0), (0, h - s), (w - s, h - s))]
        except Exception:
            pass
    return np.stack(out) if out else np.zeros((0, size, size, 3), np.uint8)


bank_file = ROBUST_DIR / "backgrounds.npz"
if bank_file.exists():                                  # same backgrounds in every session
    with np.load(bank_file) as z:
        BG_TRAIN, BG_TEST = z["train"], z["test"]
    print(f"Backgrounds reloaded from {bank_file.name}")
else:
    train_banks = [procedural_textures(200 if QUICK_RUN else 1500, seed=SEED)]
    test_banks = []
    if not QUICK_RUN:
        sources = []
        if USE_DTD:
            try:
                sources.append(("DTD textures", sorted(str(p) for p in url_download(DTD_URL, "dtd").rglob("*.jpg"))))
            except Exception as e:
                print("DTD textures unavailable:", e)
        if USE_INDOOR_SCENES:
            try:
                files = []
                for k, url in enumerate(INDOOR_URLS):
                    files += [str(p) for p in url_download(url, f"indoor_{k}").rglob("*.jpg")
                              if p.parent.name.lower() not in INDOOR_SKIP]
                sources.append(("Indoor scenes", sorted(files)))
            except Exception as e:
                print("Indoor scenes unavailable:", e)
        for k, (nm, files) in enumerate(sources):
            tr, te = split_by_category(files)
            train_banks.append(random_crops(tr, seed=SEED + k))
            test_banks.append(random_crops(te, seed=SEED + 10 + k, max_n=800))
            print(f"{nm}: {len(train_banks[-1]):,} training / {len(test_banks[-1]):,} held-out background crops")
    if rw_train_paths:
        sel = np.random.default_rng(SEED).permutation(len(rw_train_paths))[:100 if QUICK_RUN else 500]
        train_banks.append(corner_patches([rw_train_paths[i] for i in sel]))
        print(f"RealWaste floor patches: {len(train_banks[-1]):,}")
    if not test_banks:   # quick run / no downloads: hold out procedural textures from a separate seed
        test_banks = [procedural_textures(100, seed=SEED + 99)]
    BG_TRAIN, BG_TEST = np.concatenate(train_banks), np.concatenate(test_banks)
    np.savez(bank_file, train=BG_TRAIN, test=BG_TEST)
    del train_banks, test_banks
print(f"Background bank: {len(BG_TRAIN):,} for training, {len(BG_TEST):,} held out (unseen categories) for the stress test")


def composite(x, alpha, bank, scale=SCALE_RANGE):
    """x (k,H,W,3) float 0-255, alpha (k,H,W,1) 0-1: shrink/move the object and paste it on a random background."""
    b = tf.shape(x)[0]
    size = tf.cast(tf.shape(x)[1], tf.float32)
    s = tf.random.uniform([b], scale[0], scale[1])
    tx = tf.random.uniform([b]) * (1 - s) * size
    ty = tf.random.uniform([b]) * (1 - s) * size
    z = tf.zeros([b])
    transform = tf.stack([1 / s, z, -tx / s, z, 1 / s, -ty / s, z, z], axis=1)
    hard = tf.random.uniform([b, 1, 1, 1]) < 0.5                     # randomised blending: hard or soft edge
    alpha = tf.where(hard, tf.cast(alpha > 0.5, tf.float32), alpha)
    moved = tf.raw_ops.ImageProjectiveTransformV3(images=tf.concat([x, alpha], -1), transforms=transform,
                                                  output_shape=tf.shape(x)[1:3], fill_value=0.0,
                                                  interpolation="BILINEAR", fill_mode="CONSTANT")
    obj, a2 = moved[..., :3], moved[..., 3:]
    bg = tf.cast(tf.gather(bank, tf.random.uniform([b], 0, tf.shape(bank)[0], tf.int32)), tf.float32)
    bg = tf.image.random_flip_left_right(tf.image.random_flip_up_down(bg))
    bg = tf.clip_by_value(bg * tf.random.uniform([b, 1, 1, 1], 0.6, 1.2), 0, 255)
    return a2 * obj + (1 - a2) * bg

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
N_TRAIN = len(X_train)
with tf.device("/CPU:0"):                    # keep the big arrays in host memory, not on the GPU
    X_train_t = tf.constant(X_train)
    Y_train_t = tf.constant(keras.utils.to_categorical(Y_train, NUM_CLASSES))
    ROW_t = tf.constant(alpha_row)
    P_t = tf.constant(P_ROW)
    ALPHA_t = tf.constant(ALPHA[..., None] if len(ALPHA) else np.zeros((1, IMG_SIZE, IMG_SIZE, 1), np.uint8))
    BANK_t = tf.constant(BG_TRAIN)
del X_train, ALPHA


def make_train_ds(rows, swap):
    """Index-based pipeline: only the rows picked for compositing are processed (cheap on Colab's 2 CPU cores)."""
    def load(idx):
        x = tf.cast(tf.gather(X_train_t, idx), tf.float32)
        y = tf.gather(Y_train_t, idx)
        if swap:
            pos = tf.where(tf.random.uniform(tf.shape(idx)) < tf.gather(P_t, idx))          # (k, 1)
            r = tf.gather_nd(tf.gather(ROW_t, idx), pos)
            a = tf.where((r >= 0)[:, None, None, None],
                         tf.cast(tf.gather(ALPHA_t, tf.maximum(r, 0)), tf.float32) / 255.0, 1.0)  # no mask: whole picture
            x = tf.tensor_scatter_nd_update(x, pos, composite(tf.gather_nd(x, pos), a, BANK_t))
        return x, y

    rows = np.asarray(rows, np.int64)
    return (tf.data.Dataset.from_tensor_slices(rows).shuffle(len(rows), seed=SEED, reshuffle_each_iteration=True)
            .batch(BATCH_SIZE).map(load, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE))


def make_ds(x, y=None):
    y = np.zeros(len(x), "int32") if y is None else y
    ds = tf.data.Dataset.from_tensor_slices((x, keras.utils.to_categorical(y, NUM_CLASSES)))
    return ds.batch(BATCH_SIZE).map(lambda a, b: (tf.cast(a, tf.float32), b), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)


val_ds = make_ds(X_val, Y_val)
ALL_ROWS = np.arange(N_TRAIN)

xb, yb = next(iter(make_train_ds(ALL_ROWS, USE_BACKGROUND_SWAP)))
fig, axes = plt.subplots(3, 8, figsize=(16, 6.4))
for ax, img, lab in zip(axes.flat, xb.numpy()[:24], yb.numpy()[:24]):
    ax.imshow(np.clip(img, 0, 255).astype(np.uint8)); ax.axis("off")
    ax.set_title(CLASS_NAMES[int(lab.argmax())], fontsize=8)
fig.suptitle("One training batch after background swap and zoom-out (before the in-model flips, rotations and colour changes)")
plt.tight_layout(); save_fig(fig, "augmented_batch"); plt.show()

## 7. Matched-geometry stress test

For garbage-12 **test** images whose masks are usable, the object is shrunk and moved (50-90% size, fixed random seed) and pasted twice, with exactly the same size, position and edge:
* **own background**: on a plain fill in the colour of the image's own border (for these images, near-white),
* **new background**: on a held-out background from texture and room categories never used in training.

The only difference between the two copies is the background, so the drop from the first to the second, the **background gap**, measures how much a model depends on the background (after Xiao et al., 2021). The edge uses a 3 × 3 box blur, a blend not used in training. Classes with fewer than 10 stress images are left out of the stress macro F1, and the per-class counts are printed. CNN-R has trained on composites (of other images, onto other background kinds), so its gap shows the mechanism works, not that it transfers to real photos.

In [ ]:
test_df0, X_test0 = test_sets["garbage-12 test"]
cand = np.where(test_df0["white_border"].values > 0.6)[0]
masks = [object_alpha(X_test0[i]) for i in cand]
use = [k for k, (_, ok) in enumerate(masks) if ok]
stress_idx = cand[use]


def paste_fixed(x, alpha, s, tx, ty, bg):
    """Same warp as training, with GIVEN scale, offsets and backgrounds."""
    x, a = tf.constant(x, tf.float32), tf.constant(alpha, tf.float32)
    s, tx, ty = (tf.constant(v, tf.float32) for v in (s, tx, ty))
    z = tf.zeros_like(s)
    tr = tf.stack([1 / s, z, -tx / s, z, 1 / s, -ty / s, z, z], axis=1)
    m = tf.raw_ops.ImageProjectiveTransformV3(images=tf.concat([x, a], -1), transforms=tr, output_shape=tf.shape(x)[1:3],
                                              fill_value=0.0, interpolation="BILINEAR", fill_mode="CONSTANT")
    out = m[..., 3:] * m[..., :3] + (1 - m[..., 3:]) * tf.constant(bg, tf.float32)
    return np.clip(out.numpy(), 0, 255).astype(np.uint8)


def border_colour(X, frac=0.1):
    b = max(1, int(X.shape[1] * frac))
    ring = np.concatenate([X[:, :b].reshape(len(X), -1, 3), X[:, -b:].reshape(len(X), -1, 3),
                           X[:, :, :b].reshape(len(X), -1, 3), X[:, :, -b:].reshape(len(X), -1, 3)], 1)
    return np.median(ring, axis=1)[:, None, None, :].astype(np.float32)


if len(stress_idx):
    n = len(stress_idx)
    r = np.random.default_rng(SEED)
    hard = np.stack([(masks[k][0] > 127) for k in use])[..., None].astype(np.float32)
    s_alpha = tf.nn.avg_pool2d(hard, 3, 1, "SAME").numpy()                        # 3x3 box blur (not used in training)
    s = r.uniform(0.5, 0.9, n); tx = r.uniform(0, 1, n) * (1 - s) * IMG_SIZE; ty = r.uniform(0, 1, n) * (1 - s) * IMG_SIZE
    X_obj = X_test0[stress_idx]
    bg_own = np.broadcast_to(border_colour(X_obj), X_obj.shape).astype(np.float32)
    bg_new = np.clip(BG_TEST[r.integers(0, len(BG_TEST), n)].astype(np.float32) * r.uniform(0.6, 1.2, (n, 1, 1, 1)), 0, 255)
    stress_df = test_df0.iloc[stress_idx].reset_index(drop=True)
    test_sets["stress: own background"] = (stress_df, paste_fixed(X_obj, s_alpha, s, tx, ty, bg_own))
    test_sets["stress: new background"] = (stress_df, paste_fixed(X_obj, s_alpha, s, tx, ty, bg_new))
    counts = stress_df["label"].value_counts()
    print(f"Stress test: {n:,} garbage-12 test images with usable masks")
    display(counts.rename("stress images").to_frame().T)
    fig, axes = plt.subplots(3, 8, figsize=(16, 6.6))
    for j in range(min(8, n)):
        for rr, img in enumerate((X_obj[j], test_sets["stress: own background"][1][j], test_sets["stress: new background"][1][j])):
            axes[rr, j].imshow(img); axes[rr, j].axis("off")
        axes[0, j].set_title(stress_df["label"][j], fontsize=8)
    fig.suptitle("Stress test: original (top), same geometry on its own background colour (middle) and on a held-out background (bottom)")
    plt.tight_layout(); save_fig(fig, "stress_examples"); plt.show()

## 8. Models and training

Same building blocks and recipe as notebook 01 (mixed precision, class weights, label smoothing 0.1, Adam, early stopping on validation macro F1 after 12 epochs, learning rate halved after 4 flat epochs, at most 80 epochs).

| Run | Model | Training data | Background swap | Purpose |
|---|---|---|---|---|
| FCNN-R | FCNN-C architecture | all | yes | does the upgrade help an FCNN? (before: FCNN-C) |
| **CNN-R** | CNN-B architecture | all | yes | **main robust model** (before: CNN-B) |
| CNN-R-noswap | CNN-B architecture | all | no | effect of the background swap = CNN-R vs CNN-R-noswap |
| CNN-R-noown | CNN-B architecture | all except own photos | yes | public data + swap only = the cleanest real-photo comparison |
| ResCNN-R | small residual CNN, about 5 M weights | all | yes | optional (`TRAIN_RESCNN`) |

Every run saves a backup after each epoch to Google Drive (`BackupAndRestore`), so a Colab disconnect only loses the current epoch: run the notebook again and training continues.

In [ ]:
CLASS_WEIGHT_ALL = None


def augmentation_block():
    return keras.Sequential([
        layers.RandomFlip("horizontal", dtype="float32"),
        layers.RandomRotation(0.05, fill_mode="reflect", dtype="float32"),
        layers.RandomZoom((-0.15, 0.15), fill_mode="reflect", dtype="float32"),
        layers.RandomTranslation(0.1, 0.1, fill_mode="reflect", dtype="float32"),
        layers.RandomContrast(0.2, dtype="float32"),
        layers.RandomBrightness(0.15, value_range=(0, 255), dtype="float32"),
    ], name="augmentation")


def resizing_layer(size):
    return layers.AveragePooling2D(pool_size=IMG_SIZE // size, dtype="float32", name=f"downsample_{size}")


def input_block(input_size, augment, mean, var):
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    x = augmentation_block()(inp) if augment else inp
    if input_size != IMG_SIZE:
        x = resizing_layer(input_size)(x)
    x = layers.Normalization(mean=list(mean), variance=list(var), dtype="float32", name="standardise")(x)
    return inp, x


def head(x, dropout=0.4):
    x = layers.Dropout(dropout, name="head_dropout1")(x)
    x = layers.Dense(256, use_bias=False, kernel_initializer="he_normal", name="head_dense")(x)
    x = layers.BatchNormalization(name="head_bn")(x)
    x = layers.Activation("relu", name="head_relu")(x)
    x = layers.Dropout(dropout, name="head_dropout2")(x)
    return layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32", name="class_probs")(x)


def build_fcnn(name, input_size=64, hidden=(1024, 512, 256), dropout=0.4, l2=1e-4, augment=True, mean=None, var=None):
    inp, x = input_block(input_size, augment, MEAN if mean is None else mean, VAR if var is None else var)
    x = layers.Flatten(name="flatten")(x)
    for i, units in enumerate(hidden, 1):
        x = layers.Dense(units, use_bias=False, kernel_initializer="he_normal",
                         kernel_regularizer=keras.regularizers.l2(l2), name=f"dense{i}")(x)
        x = layers.BatchNormalization(name=f"bn{i}")(x)
        x = layers.Activation("relu", name=f"relu{i}")(x)
        x = layers.Dropout(dropout, name=f"dropout{i}")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32", name="class_probs")(x)
    return keras.Model(inp, out, name=name)


def conv_bn_relu(x, filters, name, l2, stride=1):
    x = layers.Conv2D(filters, 3, strides=stride, padding="same", use_bias=False, kernel_initializer="he_normal",
                      kernel_regularizer=keras.regularizers.l2(l2) if l2 else None, name=f"{name}_conv")(x)
    x = layers.BatchNormalization(name=f"{name}_bn")(x)
    return layers.Activation("relu", name=f"{name}_relu")(x)


def build_cnn(name, input_size=128, filters=(32, 64, 128, 256), block_dropout=0.1, l2=1e-4, augment=True, mean=None, var=None):
    inp, x = input_block(input_size, augment, MEAN if mean is None else mean, VAR if var is None else var)
    for b, f in enumerate(filters, 1):
        x = conv_bn_relu(x, f, f"block{b}a", l2)
        x = conv_bn_relu(x, f, f"block{b}b", l2)       # block4b_relu = Grad-CAM layer
        x = layers.MaxPooling2D(2, name=f"block{b}_pool")(x)
        x = layers.Dropout(block_dropout, name=f"block{b}_dropout")(x)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    return keras.Model(inp, head(x), name=name)


def res_block(x, filters, stride, name, l2=1e-4):
    shortcut = x
    y = conv_bn_relu(x, filters, f"{name}_1", l2, stride)
    y = layers.Conv2D(filters, 3, padding="same", use_bias=False, kernel_initializer="he_normal",
                      kernel_regularizer=keras.regularizers.l2(l2), name=f"{name}_2_conv")(y)
    y = layers.BatchNormalization(name=f"{name}_2_bn")(y)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, use_bias=False, kernel_initializer="he_normal",
                                 name=f"{name}_proj_conv")(shortcut)
        shortcut = layers.BatchNormalization(name=f"{name}_proj_bn")(shortcut)
    return layers.Activation("relu", name=f"{name}_relu")(layers.Add(name=f"{name}_add")([y, shortcut]))


def build_rescnn(name, input_size=128, widths=(48, 96, 192, 320), augment=True):
    """Small ResNet-style CNN (identity shortcuts as in He et al., 2016), trained from scratch."""
    inp, x = input_block(input_size, augment, MEAN, VAR)
    x = conv_bn_relu(x, 32, "stem", 1e-4)
    for s, w in enumerate(widths, 1):
        x = res_block(x, w, 2, f"stage{s}a")
        x = res_block(x, w, 1, f"stage{s}b")
    x = layers.Activation("linear", dtype="float32", name="res_out")(x)   # Grad-CAM layer
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    return keras.Model(inp, head(x, 0.3), name=name)


GRADCAM_LAYERS = {"CNN-R": "block4b_relu", "CNN-R-noswap": "block4b_relu", "CNN-R-noown": "block4b_relu",
                  "ResCNN-R": "res_out"}
not_own = np.flatnonzero(train_df["source"].values != "own photos")
ROBUST_RUNS = {   # name: (builder, training rows, background swap)
    "FCNN-R": (lambda: build_fcnn("FCNN-R", input_size=64), ALL_ROWS, USE_BACKGROUND_SWAP),
    "CNN-R": (lambda: build_cnn("CNN-R"), ALL_ROWS, USE_BACKGROUND_SWAP),
}
if RUN_ABLATIONS:
    ROBUST_RUNS["CNN-R-noswap"] = (lambda: build_cnn("CNN-R-noswap"), ALL_ROWS, False)
    if len(own):
        ROBUST_RUNS["CNN-R-noown"] = (lambda: build_cnn("CNN-R-noown"), not_own, USE_BACKGROUND_SWAP)
if TRAIN_RESCNN:
    ROBUST_RUNS["ResCNN-R"] = (lambda: build_rescnn("ResCNN-R"), ALL_ROWS, USE_BACKGROUND_SWAP)
for n, (b, rows, sw) in ROBUST_RUNS.items():
    print(f"{n:13s} weights: {b().count_params():>12,} | training rows: {len(rows):,} | background swap: {sw}")

In [ ]:
class EpochTimer(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.times = []

    def on_epoch_begin(self, epoch, logs=None):
        self._t0 = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        self.times.append(time.perf_counter() - self._t0)


RESULTS, MODELS = {}, {}
g12_val = (val_df["source"] == "garbage-12").values


def run_experiment(name, builder, rows, swap, lr=1e-3):
    w_path, rec_path = MODELS_DIR / f"{name}.weights.h5", LOGS_DIR / f"{name}_record.json"
    best_path, log_path = MODELS_DIR / f"{name}.best.weights.h5", LOGS_DIR / f"{name}_log.csv"
    if SKIP_TRAINED and w_path.exists() and rec_path.exists():
        model = builder()
        model.load_weights(w_path)
        model.compile(jit_compile=False)
        RESULTS[name] = json.loads(rec_path.read_text())
        print(f"[{name}] loaded from {w_path.name} (already trained)")
        return model
    keras.backend.clear_session()               # also resets the dtype policy in Keras 3...
    if GPUS:
        keras.mixed_precision.set_global_policy("mixed_float16")   # ...so switch mixed precision back on
    keras.utils.set_random_seed(SEED)
    gpu_base = 0
    if GPUS:
        tf.config.experimental.reset_memory_stats("GPU:0")
        gpu_base = tf.config.experimental.get_memory_info("GPU:0")["current"]
    model = builder()
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
                  metrics=["accuracy", keras.metrics.F1Score(average="macro", name="macro_f1")],
                  jit_compile=False)
    best_so_far = None
    if (MODELS_DIR / f"{name}_backup").exists() and log_path.exists():     # resuming after a disconnect
        try:
            best_so_far = float(pd.read_csv(log_path)["val_macro_f1"].max())
        except Exception:
            best_so_far = None
    else:                                                                   # fresh start: clear old files
        log_path.unlink(missing_ok=True)
        best_path.unlink(missing_ok=True)
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=Y_train[rows])))
    timer = EpochTimer()
    callbacks = [
        keras.callbacks.BackupAndRestore(str(MODELS_DIR / f"{name}_backup"), delete_checkpoint=True),
        keras.callbacks.ModelCheckpoint(str(best_path), monitor="val_macro_f1", mode="max", save_best_only=True,
                                        save_weights_only=True, initial_value_threshold=best_so_far),
        keras.callbacks.EarlyStopping(monitor="val_macro_f1", mode="max", patience=EARLY_STOP_PATIENCE,
                                      restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
        keras.callbacks.CSVLogger(str(log_path), append=True),
        timer,
    ]
    epochs = 2 if QUICK_RUN else MAX_EPOCHS
    print(f"\n=== Training {name} ({model.count_params():,} weights, {len(rows):,} rows, swap={swap}, up to {epochs} epochs) ===")
    t0 = time.perf_counter()
    model.fit(make_train_ds(rows, swap), validation_data=val_ds, epochs=epochs, class_weight=cw,
              callbacks=callbacks, verbose=2)
    total = time.perf_counter() - t0
    if best_path.exists():
        model.load_weights(best_path)          # best epoch, also correct after a resumed run
    h = pd.read_csv(log_path).to_dict("list")
    pv = model.predict(make_ds(X_val), verbose=0).argmax(1)
    record = {"name": name, "dtype_policy": model.dtype_policy.name, "params": int(model.count_params()),
              "rows": int(len(rows)), "background_swap": bool(swap),
              "epochs_run": len(h["loss"]), "hit_epoch_cap": len(h["loss"]) >= epochs,
              "best_epoch": int(np.argmax(h["val_macro_f1"])) + 1, "val_macro_f1": float(max(h["val_macro_f1"])),
              "val_macro_f1_garbage12_only": float(f1_score(Y_val[g12_val], pv[g12_val], average="macro", zero_division=0)),
              "sec_per_epoch": float(np.median(timer.times[1:] if len(timer.times) > 1 else timer.times)) if timer.times else None,
              "train_minutes": total / 60,
              "peak_gpu_mb": (tf.config.experimental.get_memory_info("GPU:0")["peak"] - gpu_base) / 2**20 if GPUS else None,
              "history": h}
    model.save_weights(w_path)
    rec_path.write_text(json.dumps(record))
    RESULTS[name] = record
    print(f"[{name}] done in {total / 60:.1f} min ({record['dtype_policy']}), best val macro F1 {record['val_macro_f1']:.3f} "
          f"(garbage-12 val only: {record['val_macro_f1_garbage12_only']:.3f})")
    return model


for name, (builder, rows, swap) in ROBUST_RUNS.items():
    MODELS[name] = run_experiment(name, builder, rows, swap)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for n in ROBUST_RUNS:
    h = RESULTS[n]["history"]
    line, = axes[0].plot(h["val_loss"], label=n)
    axes[0].plot(h["loss"], ls="--", color=line.get_color(), alpha=0.6)
    line, = axes[1].plot(h["val_macro_f1"], label=n)
    axes[1].plot(h["macro_f1"], ls="--", color=line.get_color(), alpha=0.6)
axes[0].set(title="Loss (solid = validation, dashed = training)", xlabel="epoch")
axes[1].set(title="Macro F1 (solid = validation, dashed = training)", xlabel="epoch")
axes[1].legend(fontsize=8)
plt.tight_layout(); save_fig(fig, "curves_robust"); plt.show()

## 9. Before vs after

**"Before"** = notebook 01's CNN-B and FCNN-C, rebuilt here with notebook 01's own normalisation and saved weights, so they are exactly the same networks; they are the same architectures as CNN-R and FCNN-R. Notebook 01's exported best models are shown too when they are different runs.

Reading guide:
* **garbage-12 test**: like-for-like for every model.
* **extra test sets**: in-domain for the "after" models (they trained on these sources), so a gain there is expected and is *not* evidence of robustness. A second table scores them only among the classes each source contains.
* **stress tests**: the background gap (own background minus new background) is the robustness measure.
* Macro F1 is the main metric. A single training seed per model: notebook 01's seed repeats (`logs/seed_variation.csv`) give the noise level.

In [ ]:
def nb1_model(run, builder):
    """Rebuild a notebook-01 run with notebook 01's normalisation and load its saved weights."""
    keras.mixed_precision.set_global_policy("float32")
    m = builder()
    m.load_weights(MODELS_01 / f"{run}.weights.h5")
    m.compile(jit_compile=False)
    if GPUS:
        keras.mixed_precision.set_global_policy("mixed_float16")
    return m


ALL = {"before: CNN-B": nb1_model("CNN-B", lambda: build_cnn("CNN-B", mean=B_MEAN, var=B_VAR)),
       "before: FCNN-C": nb1_model("FCNN-C", lambda: build_fcnn("FCNN-C", input_size=64, mean=B_MEAN, var=B_VAR))}
for tag in ("cnn", "fcnn"):
    run = META["models"][tag]["run"]
    if f"before: {run}" not in ALL:
        m = keras.models.load_model(EXPORT_DIR / META["models"][tag]["file"], compile=False)
        m.compile(jit_compile=False)
        ALL[f"before: {run} (notebook 01 best)"] = m
ALL.update({f"after: {n}": m for n, m in MODELS.items()})


def predict_probs(model, x):
    return model.predict(make_ds(x), verbose=0)


def eval_labels(set_name, y):
    present = sorted(set(y))
    if set_name.startswith("stress"):
        present = [k for k in present if (y == k).sum() >= 10]
    return present


PROBS = {m: {} for m in ALL}
rows = []
for set_name, (df_, X_) in test_sets.items():
    y = df_["y"].values.astype(int)
    labs = eval_labels(set_name, y)
    keep = np.isin(y, labs)
    for m_name, model in ALL.items():
        p = predict_probs(model, X_)
        PROBS[m_name][set_name] = p
        pred = p.argmax(1)
        rows.append({"model": m_name, "test set": set_name, "images": int(keep.sum()),
                     "macro F1": f1_score(y[keep], pred[keep], labels=labs, average="macro", zero_division=0),
                     "balanced acc": balanced_accuracy_score(y[keep], pred[keep]),
                     "accuracy": accuracy_score(y[keep], pred[keep])})
res = pd.DataFrame(rows)
res.to_csv(LOGS_DIR / "before_after_all_tests.csv", index=False)
table = res.pivot(index="model", columns="test set", values="macro F1")[list(test_sets)].loc[list(ALL)]
print("Macro F1 (stress sets: classes with at least 10 images only)")
display(table.style.format("{:.3f}").background_gradient(axis=0, cmap="Greens"))

if "stress: new background" in test_sets:
    labs = eval_labels("stress", stress_df["y"].values)
    print("Stress classes used:", ", ".join(f"{CLASS_NAMES[k]} ({int((stress_df['y'] == k).sum())})" for k in labs))
    gap = pd.DataFrame({
        "accuracy, own background": res[res["test set"] == "stress: own background"].set_index("model")["accuracy"],
        "accuracy, new background": res[res["test set"] == "stress: new background"].set_index("model")["accuracy"]}).loc[list(ALL)]
    gap["background gap"] = gap["accuracy, own background"] - gap["accuracy, new background"]
    gap.to_csv(LOGS_DIR / "background_gap.csv")
    display(gap.style.format("{:.3f}"))

# extras scored only among the classes each source contains (removes the "this source has no batteries" advantage)
rows_r = []
for set_name in [k for k in test_sets if "in-domain" in k]:
    y = test_sets[set_name][0]["y"].values.astype(int)
    present = np.array(sorted(set(y)))
    for m_name in ALL:
        pred = present[PROBS[m_name][set_name][:, present].argmax(1)]
        rows_r.append({"model": m_name, "test set": set_name,
                       "macro F1, own classes only": f1_score(y, pred, labels=present, average="macro", zero_division=0)})
if rows_r:
    display(pd.DataFrame(rows_r).pivot(index="model", columns="test set", values="macro F1, own classes only")
            .loc[list(ALL)].style.format("{:.3f}"))

fig, ax = plt.subplots(figsize=(13, 4.5))
w = 0.8 / len(ALL)
xpos = np.arange(len(test_sets))
for k, m_name in enumerate(ALL):
    ax.bar(xpos + k * w - 0.4 + w / 2, table.loc[m_name].values, w, label=m_name,
           color=plt.get_cmap("tab10")(k), alpha=0.5 if m_name.startswith("before") else 1.0)
ax.set_xticks(xpos, [t.replace(" (in-domain for 'after')", "\n(in-domain)") for t in test_sets], rotation=0, fontsize=8)
ax.set_ylim(0, 1); ax.set_ylabel("macro F1")
ax.set_title("Before (faded) vs after, on every test set"); ax.legend(fontsize=7, ncol=3)
plt.tight_layout(); save_fig(fig, "before_after"); plt.show()

if "RealWaste test (in-domain for 'after')" in test_sets and "after: CNN-R" in ALL:
    y = test_sets["RealWaste test (in-domain for 'after')"][0]["y"].values.astype(int)
    labs = sorted(set(y))
    rec_ = recall_score(y, PROBS["after: CNN-R"]["RealWaste test (in-domain for 'after')"].argmax(1), labels=labs, average=None, zero_division=0)
    display(pd.Series(rec_, index=[CLASS_NAMES[k] for k in labs], name="CNN-R recall on RealWaste test").to_frame().T.style.format("{:.2f}"))

**Is the change real?** Paired comparisons on the same images: McNemar's exact test on accuracy and a paired, class-stratified bootstrap interval on macro F1. The primary pair is CNN-B → CNN-R (same network). CNN-R-noswap → CNN-R isolates the background swap; CNN-B → CNN-R-noown is "public data + background swap" without our own photos.

In [ ]:
def compare(a, b, set_name, n_boot=2000):
    y = test_sets[set_name][0]["y"].values.astype(int)
    labs = eval_labels(set_name, y)
    keep = np.isin(y, labs)
    y = y[keep]
    pa, pb = PROBS[a][set_name].argmax(1)[keep], PROBS[b][set_name].argmax(1)[keep]
    n01, n10 = int(np.sum((pa == y) & (pb != y))), int(np.sum((pa != y) & (pb == y)))
    p_mc = binomtest(min(n01, n10), n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
    mf1 = lambda t, p: f1_score(t, p, average="macro", labels=labs, zero_division=0)
    r = np.random.default_rng(SEED)
    by_cls = [np.where(y == k)[0] for k in labs]
    diffs = [mf1(y[i], pb[i]) - mf1(y[i], pa[i])
             for i in (np.concatenate([r.choice(ix, len(ix)) for ix in by_cls]) for _ in range(n_boot))]
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return {"comparison": f"{a.split(': ')[1]} → {b.split(': ')[1]}", "test set": set_name,
            "macro F1 change": mf1(y, pb) - mf1(y, pa), "95% CI low": lo, "95% CI high": hi,
            "only before right": n01, "only after right": n10, "McNemar exact p": p_mc}


PAIRS = [("before: CNN-B", "after: CNN-R"), ("after: CNN-R-noswap", "after: CNN-R"),
         ("before: CNN-B", "after: CNN-R-noown"), ("before: FCNN-C", "after: FCNN-R")]
cmp_df = pd.DataFrame([compare(a, b, s) for a, b in PAIRS if a in ALL and b in ALL for s in test_sets])
cmp_df.to_csv(LOGS_DIR / "paired_tests.csv", index=False)
display(cmp_df.style.format({"macro F1 change": "{:+.3f}", "95% CI low": "{:+.3f}", "95% CI high": "{:+.3f}",
                             "McNemar exact p": "{:.1e}"}))

**Where do the models look?** Grad-CAM on the new-background stress images: CNN-B (before) vs CNN-R (after).

In [ ]:
def grad_cam(model, img_uint8, layer_name):
    out = model.get_layer("class_probs")
    grad_model = keras.Model(model.inputs, [model.get_layer(layer_name).output, model.get_layer("head_relu").output])
    x = tf.convert_to_tensor(img_uint8[None].astype("float32"))
    with tf.GradientTape() as tape:
        fmap, h = grad_model(x, training=False)
        logits = tf.matmul(tf.cast(h, tf.float32), tf.convert_to_tensor(out.kernel)) + tf.convert_to_tensor(out.bias)
        c = int(tf.argmax(logits[0]))
        score = logits[:, c]
    grads = tf.cast(tape.gradient(score, fmap), tf.float32)
    cam = tf.nn.relu(tf.reduce_sum(tf.cast(fmap, tf.float32) * tf.reduce_mean(grads, axis=(1, 2))[:, None, None, :], -1))[0].numpy()
    return cam / (cam.max() + 1e-8), c


def overlay(img_uint8, cam, alpha=0.45):
    heat = np.asarray(Image.fromarray(np.uint8(cam * 255)).resize(img_uint8.shape[:2][::-1], Image.BILINEAR)) / 255.0
    return np.clip((1 - alpha) * img_uint8 / 255.0 + alpha * plt.get_cmap("jet")(heat)[..., :3], 0, 1)


def float32_copy(model, builder):
    keras.mixed_precision.set_global_policy("float32")
    clone = builder()
    clone.set_weights(model.get_weights())
    if GPUS:
        keras.mixed_precision.set_global_policy("mixed_float16")
    return clone


if "stress: new background" in test_sets and "CNN-R" in MODELS:
    cam_after = float32_copy(MODELS["CNN-R"], ROBUST_RUNS["CNN-R"][0])
    sdf, Xs = test_sets["stress: new background"]
    pick = np.random.default_rng(SEED).choice(len(Xs), min(8, len(Xs)), replace=False)
    fig, axes = plt.subplots(3, len(pick), figsize=(2.2 * len(pick), 7.2), squeeze=False)
    for j, i in enumerate(pick):
        axes[0, j].imshow(Xs[i]); axes[0, j].set_title(f"true: {sdf['label'][i]}", fontsize=8)
        for r_, (mdl, tag) in enumerate(((ALL["before: CNN-B"], "before"), (cam_after, "after")), 1):
            cam, c = grad_cam(mdl, Xs[i], "block4b_relu")
            axes[r_, j].imshow(overlay(Xs[i], cam))
            axes[r_, j].set_title(f"{tag}: {CLASS_NAMES[c]}", fontsize=8, color="green" if c == sdf["y"][i] else "crimson")
    for ax in axes.flat:
        ax.axis("off")
    fig.suptitle("Grad-CAM on held-out backgrounds: CNN-B before (middle) vs CNN-R after (bottom)")
    plt.tight_layout(); save_fig(fig, "gradcam_before_after"); plt.show()

## 10. Compute used

In [ ]:
comp = pd.DataFrame([{"run": n, "weights": r["params"], "rows": r["rows"], "swap": r["background_swap"],
                      "policy": r["dtype_policy"], "epochs": r["epochs_run"], "sec/epoch": r["sec_per_epoch"],
                      "train_min": r["train_minutes"], "peak_GPU_MB": r["peak_gpu_mb"],
                      "val macro F1": r["val_macro_f1"], "val macro F1 (garbage-12 only)": r["val_macro_f1_garbage12_only"]}
                     for n, r in RESULTS.items()]).set_index("run")
comp["peak_GPU_MB"] = pd.to_numeric(comp["peak_GPU_MB"], errors="coerce")
comp["sec/epoch"] = pd.to_numeric(comp["sec/epoch"], errors="coerce")
comp.to_csv(LOGS_DIR / "compute_robust.csv")
print(f"Hardware: {GPU_NAME} | training images per epoch (all rows): {N_TRAIN:,}")
display(comp.style.format({"weights": "{:,}", "rows": "{:,}", "sec/epoch": "{:.1f}", "train_min": "{:.1f}", "peak_GPU_MB": "{:.0f}",
                           "val macro F1": "{:.3f}", "val macro F1 (garbage-12 only)": "{:.3f}"}, na_rep="n/a"))

## 11. Export the robust models for the demo

CNN-R (the main robust model), CNN-R-noown and FCNN-R are saved next to notebook 01's models. Notebook 02 then shows them side by side and uses CNN-R as its main answer. The "not sure" threshold for CNN-R is chosen on validation data, as before.

In [ ]:
p_val = predict_probs(MODELS["CNN-R"], X_val)
conf_v, ok_v = p_val.max(1), p_val.argmax(1) == Y_val
grid = np.round(np.arange(0.30, 0.91, 0.01), 2)
sel = np.array([ok_v[conf_v >= t].mean() if (conf_v >= t).any() else np.nan for t in grid])
hits = np.where(sel >= 0.90)[0]
UNSURE_ROBUST = float(grid[hits[0]]) if len(hits) else 0.5

keras.mixed_precision.set_global_policy("float32")
for tag, name in (("cnn_robust", "CNN-R"), ("cnn_robust_noown", "CNN-R-noown"), ("fcnn_robust", "FCNN-R")):
    if name not in MODELS:
        META["models"].pop(tag, None)
        continue
    clean = ROBUST_RUNS[name][0]()
    clean.set_weights(MODELS[name].get_weights())
    path = EXPORT_DIR / f"best_{tag}.keras"
    clean.save(path)
    sc = res[(res["model"] == f"after: {name}") & (res["test set"] == "garbage-12 test")].iloc[0]
    META["models"][tag] = {"run": name, "file": path.name, "test_macro_f1": float(sc["macro F1"]),
                           "test_balanced_acc": float(sc["balanced acc"]), "test_accuracy": float(sc["accuracy"]),
                           "gradcam_layer": GRADCAM_LAYERS.get(name)}
    print(f"Saved {name} -> {path} ({path.stat().st_size / 2**20:.1f} MB)")
if GPUS:
    keras.mixed_precision.set_global_policy("mixed_float16")
META["unsure_below_robust"] = UNSURE_ROBUST
(EXPORT_DIR / "meta.json").write_text(json.dumps(META, indent=2))
shutil.make_archive(str(OUT_DIR / "waste_models_export"), "zip", EXPORT_DIR)
print(f"'Not sure' threshold for CNN-R: {UNSURE_ROBUST:.2f}. meta.json updated; open notebook 02 next.")

### What the report can claim
* **Garbage-12 test** (like-for-like): CNN-B → CNN-R macro F1 change with its 95% interval (paired table).
* **Background dependence**: the background gap of CNN-B vs CNN-R on the matched stress test. CNN-R trained on composites made the same way (with other images and other background kinds), so this shows the mechanism works, not that it transfers.
* **Extra test sets**: in-domain for the new models; report, do not claim as robustness.
* **Real photos** (notebook 02, fresh photos in places not used for training): the primary result, as a paired item-level difference with its interval. CNN-R-noown isolates "public data + background swap"; CNN-R adds our own photos.
* One seed per model; with about 36 photographed items only differences of roughly 15 percentage points or more can be detected.